# 🚗 Matriculator · Pseudocódigo (Paso 3)

**Responsable:** Julen Altuna · **Lenguaje elegido para IA:** Python

> ⚠️ No se implementa ni se entrena ningún modelo. Este pseudocódigo describe **cómo funcionaría** el componente de IA.
> Para poder ejecutarlo sin modelo real, `cargar_modelo()` y `cargar_ocr()` devuelven **simulaciones**.

El pseudocódigo (20–50 líneas) debe mostrar:

| Parte | Dónde está |
|---|---|
| 🟢 **Entrada** | `leer_entrada()` – JSON con la ruta de la imagen y la cámara |
| 🔵 **Funciones principales** | `validar_imagen()`, `preparar()`, `leer_matricula()` |
| 🟠 **Condición / control de errores** | `if` de formato/tamaño, umbral de confianza, `try/except` |
| 🩷 **Salida** | diccionario `resultado` + registro CSV sin guardar la imagen |

✏️ *TODO Julen: revisar, comentar con vuestras palabras y ajustar al diagrama de flujo de la web.*

## 1 · Simulaciones (sustituyen al modelo real)

In [1]:
import random

class DetectorSimulado:                      # simula un detector tipo YOLO
    def detectar(self, imagen):
        return [{"caja": (212, 318, 398, 362), "confianza": random.uniform(0.4, 0.99)}]

class OCRSimulado:                           # simula un OCR
    def leer(self, recorte):
        return " 0000-xxx "

cargar_modelo = lambda ruta: DetectorSimulado()
cargar_ocr = lambda idioma: OCRSimulado()

## 2 · Pseudocódigo del componente de IA

In [2]:
import json, csv, re, os
from datetime import datetime

UMBRAL_CONFIANZA = 0.80                       # por debajo -> revisión humana
FORMATOS = (".jpg", ".png")
PATRON = re.compile(r"^\d{4}[A-Z]{3}$")       # formato de matrícula española

def leer_entrada(ruta_json):                  # 🟢 ENTRADA
    with open(ruta_json, encoding="utf-8") as f:
        return json.load(f)

def validar_imagen(ruta):                     # 🔵 FUNCIÓN + 🟠 ERRORES
    if not ruta.lower().endswith(FORMATOS):
        raise ValueError("Formato de imagen no válido")
    return {"ruta": ruta}                    # aquí iría cv2.imread(ruta)

def preparar(imagen):                         # 🔵 redimensionar y normalizar
    return imagen                            # cv2.resize(...) / 255.0

def leer_matricula(imagen, detector, ocr):    # 🔵 detección + OCR
    cajas = detector.detectar(imagen)
    if not cajas:
        return None, 0.0
    mejor = max(cajas, key=lambda c: c["confianza"])
    texto = ocr.leer(imagen)                  # recorte de mejor["caja"]
    limpio = re.sub(r"[^0-9A-Z]", "", texto.upper())
    return limpio, round(mejor["confianza"], 2)

def procesar(entrada, detector, ocr):
    try:                                      # 🟠 control de errores
        imagen = preparar(validar_imagen(entrada["ruta_imagen"]))
        matricula, conf = leer_matricula(imagen, detector, ocr)
        if matricula is None or conf < UMBRAL_CONFIANZA or not PATRON.match(matricula):
            resultado = {"estado": "REVISION_HUMANA", "matricula": matricula, "confianza": conf}
        else:
            resultado = {"estado": "OK", "matricula": matricula, "confianza": conf}
    except ValueError as e:
        resultado = {"estado": "ERROR", "mensaje": str(e)}
    resultado["camara_id"] = entrada["camara_id"]
    resultado["fecha"] = datetime.now().isoformat(timespec="seconds")
    return resultado                          # 🩷 SALIDA

def guardar_registro(ruta_csv, resultado):   # 🩷 registro mínimo, sin imagen
    nuevo = not os.path.exists(ruta_csv)
    with open(ruta_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["fecha", "camara_id", "estado", "matricula", "confianza", "mensaje"])
        if nuevo: w.writeheader()
        w.writerow(resultado)

## 3 · Ejecución de prueba con datos ficticios

In [3]:
# Entrada ficticia (en la app llegaría como JSON desde el servidor)
entradas = [
    {"ruta_imagen": "pruebas/imagen_001.jpg", "camara_id": "CAM-01"},
    {"ruta_imagen": "pruebas/imagen_002.gif", "camara_id": "CAM-02"},   # formato no válido
]
detector, ocr = cargar_modelo("modelos/detector.pt"), cargar_ocr("es")
for e in entradas:
    print(procesar(e, detector, ocr))

{'estado': 'REVISION_HUMANA', 'matricula': '0000XXX', 'confianza': 0.66, 'camara_id': 'CAM-01', 'fecha': '2026-09-22T19:35:14'}
{'estado': 'ERROR', 'mensaje': 'Formato de imagen no válido', 'camara_id': 'CAM-02', 'fecha': '2026-09-22T19:35:14'}


## 4 · Explicación

✏️ *TODO Julen: explicar en 5-8 líneas qué hace cada función, por qué hay un umbral de confianza y en qué momento interviene la revisión humana (relacionarlo con la etapa 7 del flujo de la web).*